# Loan Approval Prediction - Member 3: Model Building, Evaluation, Tuning & Deployment

**Member 3 Objectives:**
1. Data Preprocessing & Train-Test Split
2. Model Training (Logistic Regression, Decision Tree, Random Forest)
3. Model Evaluation & Performance Metrics (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
4. Hyperparameter Tuning using GridSearchCV
5. Feature Importance Analysis
6. Model Serialization (`joblib` export)
7. Sample Inference & Prediction Function
8. Streamlit Web Application (`app.py`)


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully!")

## 1. Load Dataset & Data Preprocessing

In [ ]:
file_path = "data/Loan Eligibility Prediction.csv"
if not os.path.exists(file_path):
    file_path = "Loan Eligibility Prediction.csv"

df = pd.read_csv(file_path)
print("Original Dataset Shape:", df.shape)

# Drop duplicates
df = df.drop_duplicates()

# Clean target
df_model = df.dropna(subset=["Loan_Status"]).copy()
df_model["Loan_Status"] = df_model["Loan_Status"].astype(str).str.strip().str.upper()
df_model["Loan_Status"] = df_model["Loan_Status"].map({"Y": 1, "N": 0})

# Impute missing numerical & categorical values
num_cols = df_model.select_dtypes(include=["int64", "float64"]).columns
for col in num_cols:
    if col != "Loan_Status":
        df_model[col] = df_model[col].fillna(df_model[col].median())

cat_cols = df_model.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    if col != "Customer_ID":
        df_model[col] = df_model[col].fillna(df_model[col].mode()[0])

# Drop ID column if present
if "Customer_ID" in df_model.columns:
    df_model = df_model.drop(columns=["Customer_ID"])

X = df_model.drop(columns=["Loan_Status"])
y = df_model["Loan_Status"]

# One-Hot Encoding
categorical_cols = X.select_dtypes(include=["object", "string"]).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True).astype(int)

print("Encoded Features Shape (X):", X.shape)
print("Target Shape (y):", y.shape)

## 2. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set Shape:", X_train.shape)
print("Testing Set Shape:", X_test.shape)

## 3. Train Baseline Machine Learning Models

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

decision_tree = DecisionTreeClassifier(random_state=42, max_depth=5)
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit models
logistic_model.fit(X_train, y_train)
decision_tree.fit(X_train, y_train)
random_forest.fit(X_train, y_train)

print("Models Trained Successfully!")

## 4. Model Predictions & Evaluation Metrics

In [ ]:
y_pred_logistic = logistic_model.predict(X_test)
y_pred_tree = decision_tree.predict(X_test)
y_pred_forest = random_forest.predict(X_test)

logistic_accuracy = accuracy_score(y_test, y_pred_logistic)
tree_accuracy = accuracy_score(y_test, y_pred_tree)
forest_accuracy = accuracy_score(y_test, y_pred_forest)

print(f"Logistic Regression Accuracy: {logistic_accuracy * 100:.2f}%")
print(f"Decision Tree Accuracy      : {tree_accuracy * 100:.2f}%")
print(f"Random Forest Accuracy      : {forest_accuracy * 100:.2f}%")

In [ ]:
models_predictions = {
    "Logistic Regression": y_pred_logistic,
    "Decision Tree": y_pred_tree,
    "Random Forest": y_pred_forest
}

comparison = []
for name, predictions in models_predictions.items():
    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1-Score": f1_score(y_test, predictions)
    })

comparison_df = pd.DataFrame(comparison).sort_values(by="Accuracy", ascending=False)
display(comparison_df)

## 5. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", grid_search.best_params_)
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_ * 100:.2f}%")

best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned Random Forest Test Accuracy: {tuned_accuracy * 100:.2f}%")

In [ ]:
if tuned_accuracy >= forest_accuracy:
    final_model = best_rf
    final_accuracy = tuned_accuracy
    final_predictions = y_pred_tuned
    print("Selected Final Model: Tuned Random Forest Classifier")
else:
    final_model = random_forest
    final_accuracy = forest_accuracy
    final_predictions = y_pred_forest
    print("Selected Final Model: Original Random Forest Classifier")

print("\n--- Classification Report ---")
print(classification_report(y_test, final_predictions))

## 6. Confusion Matrix & Feature Importance

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, final_predictions, cmap="Blues")
plt.title("Final Random Forest - Confusion Matrix")
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": final_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10, 6))
top_features = feature_importance.head(10)
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1], color="skyblue")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importance - Random Forest")
plt.show()

## 7. Save Model & Feature Columns

In [ ]:
joblib.dump(final_model, "loan_eligibility_random_forest.pkl")
feature_columns = X_train.columns.tolist()
joblib.dump(feature_columns, "loan_feature_columns.pkl")

print("Model file created: loan_eligibility_random_forest.pkl")
print("Feature file created: loan_feature_columns.pkl")

## 8. Sample Inference Function

In [ ]:
def predict_loan(
    gender, married, dependents, education, self_employed,
    applicant_income, coapplicant_income, loan_amount, loan_amount_term,
    credit_history, property_area
):
    customer = pd.DataFrame({
        "Gender": [gender],
        "Married": [married],
        "Dependents": [dependents],
        "Education": [education],
        "Self_Employed": [self_employed],
        "Applicant_Income": [applicant_income],
        "Coapplicant_Income": [coapplicant_income],
        "Loan_Amount": [loan_amount],
        "Loan_Amount_Term": [loan_amount_term],
        "Credit_History": [credit_history],
        "Property_Area": [property_area]
    })

    customer_encoded = pd.get_dummies(customer, columns=customer.select_dtypes(include=["object", "string"]).columns, drop_first=True)
    customer_encoded = customer_encoded.reindex(columns=feature_columns, fill_value=0).astype(int)

    pred = final_model.predict(customer_encoded)[0]
    prob = final_model.predict_proba(customer_encoded)[0][1]

    status = "Loan Approved" if pred == 1 else "Loan Not Approved"
    return status, prob

status, probability = predict_loan(
    gender="Male", married="Yes", dependents="0", education="Graduate", self_employed="No",
    applicant_income=5000, coapplicant_income=2000, loan_amount=150, loan_amount_term=360,
    credit_history=1, property_area="Urban"
)

print("Sample Prediction Result:", status)
print(f"Approval Probability   : {probability * 100:.2f}%")

## 9. Export Streamlit App (`app.py`)

In [ ]:
app_code = '''import streamlit as st
import pandas as pd
import joblib

# Load model and feature columns
model = joblib.load("loan_eligibility_random_forest.pkl")
feature_columns = joblib.load("loan_feature_columns.pkl")

def predict_loan(gender, married, dependents, education, self_employed, applicant_income, coapplicant_income, loan_amount, loan_amount_term, credit_history, property_area):
    customer = pd.DataFrame({
        "Gender": [gender],
        "Married": [married],
        "Dependents": [dependents],
        "Education": [education],
        "Self_Employed": [self_employed],
        "Applicant_Income": [applicant_income],
        "Coapplicant_Income": [coapplicant_income],
        "Loan_Amount": [loan_amount],
        "Loan_Amount_Term": [loan_amount_term],
        "Credit_History": [credit_history],
        "Property_Area": [property_area]
    })
    categorical_columns = customer.select_dtypes(include=["object", "string"]).columns
    customer_encoded = pd.get_dummies(customer, columns=categorical_columns, drop_first=True)
    customer_encoded = customer_encoded.reindex(columns=feature_columns, fill_value=0).astype(int)
    prediction = model.predict(customer_encoded)[0]
    probability = model.predict_proba(customer_encoded)[0][1]
    status = "Loan Approved" if prediction == 1 else "Loan Not Approved"
    return status, probability

st.set_page_config(page_title="Loan Eligibility Prediction", page_icon="💰", layout="centered")
st.title("💰 Loan Eligibility Prediction System")
st.write("Enter applicant details to predict loan approval status.")
st.divider()

col1, col2 = st.columns(2)
with col1:
    gender = st.selectbox("Gender", ["Male", "Female"])
    married = st.selectbox("Married", ["Yes", "No"])
    dependents = st.selectbox("Dependents", ["0", "1", "2", "3+"])
    education = st.selectbox("Education", ["Graduate", "Not Graduate"])
    self_employed = st.selectbox("Self Employed", ["Yes", "No"])
    property_area = st.selectbox("Property Area", ["Urban", "Semiurban", "Rural"])

with col2:
    applicant_income = st.number_input("Applicant Income", min_value=0, value=5000, step=100)
    coapplicant_income = st.number_input("Coapplicant Income", min_value=0, value=0, step=100)
    loan_amount = st.number_input("Loan Amount", min_value=0, value=150, step=10)
    loan_amount_term = st.number_input("Loan Amount Term", min_value=1, value=360, step=12)
    credit_history = st.selectbox("Credit History", [1, 0], format_func=lambda x: "Good (1)" if x == 1 else "Poor (0)")

st.divider()
if st.button("🔍 Predict Loan Eligibility", use_container_width=True):
    status, probability = predict_loan(gender, married, dependents, education, self_employed, applicant_income, coapplicant_income, loan_amount, loan_amount_term, credit_history, property_area)
    if status == "Loan Approved":
        st.success(f"✅ {status}")
    else:
        st.error(f"❌ {status}")
    st.metric("Approval Probability", f"{probability * 100:.2f}%")
    st.progress(float(probability))
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully!")